# 02: COPY INTO

**Exam objective (from Data Ingestion and Loading domain):** Use the COPY INTO command to incrementally load files from cloud object storage (ADLS/S3/GCS) into Unity-Catalog–governed tables.

**Free Edition note:** Source files are landed in a UC volume rather than external cloud storage. The COPY INTO syntax is identical regardless of storage backend; only the source path prefix differs.

**Conceptual relationship to Auto Loader:** Both COPY INTO and Auto Loader provide incremental, idempotent file ingestion to Delta tables. COPY INTO is the SQL-first, batch-only option; Auto Loader is the PySpark-first option with streaming support. Both track ingested files to prevent duplication.

In [0]:
%sql
-- Create a volume to hold source files
CREATE VOLUME IF NOT EXISTS certprep.ingestion.copy_into_files;

In [0]:
%sql
-- Create the target table. Unlike Auto Loader, COPY INTO works best when the target table already exists with a defined schema.
CREATE TABLE IF NOT EXISTS certprep.ingestion.products (
  product_id INT,
  product_name STRING,
  category STRING,
  price DECIMAL(10, 2),
  in_stock BOOLEAN
);

In [0]:
%sql
-- Verify both exist
SHOW VOLUMES IN certprep.ingestion;
SHOW TABLES IN certprep.ingestion;

Why does the target table pre-exist? This is the first meaningful difference from Auto Loader. With Auto Loader, you typically let the stream create the table on first write (inferring schema from the source data). With COPY INTO, the common pattern is to define the target table's schema explicitly first, then COPY INTO files that conform to it. COPY INTO can create the table on the fly using its COPY_OPTIONS (specifically with mergeSchema = true and certain other settings), but the explicit-schema-first pattern is the more common production pattern and what the exam will likely test.

The schema mismatch behavior: Because the schema is pre-defined, COPY INTO has clearer rules for what happens when source data doesn't match: by default, the load fails. You can opt into schema evolution with COPY_OPTIONS ('mergeSchema' = 'true'), similar to Auto Loader's write-side option.

COPY INTO typically works against a pre-defined target table. This is different from the Auto Loader pattern where the stream often creates the table on first write. The pre-defined schema gives COPY INTO clearer mismatch rules: by default, source data not matching the target schema causes the load to fail. Schema evolution is opt-in via COPY_OPTIONS.

In [0]:
import json

volume_path = "/Volumes/certprep/ingestion/copy_into_files"

In [0]:
# Clean slate
dbutils.fs.rm(volume_path, recurse=True)
dbutils.fs.mkdirs(volume_path)

In [0]:
# Generate three batches of product data, simulating daily file drops
batches = [
    [
        {"product_id": 1, "product_name": "Wireless Mouse",     "category": "Electronics", "price": 29.99,  "in_stock": True},
        {"product_id": 2, "product_name": "Mechanical Keyboard","category": "Electronics", "price": 89.99,  "in_stock": True},
    ],
    [
        {"product_id": 3, "product_name": "Coffee Mug",         "category": "Kitchen",     "price": 12.50,  "in_stock": True},
        {"product_id": 4, "product_name": "Cast Iron Skillet",  "category": "Kitchen",     "price": 45.00,  "in_stock": False},
    ],
    [
        {"product_id": 5, "product_name": "Notebook",           "category": "Office",      "price": 6.75,   "in_stock": True},
        {"product_id": 6, "product_name": "Desk Lamp",          "category": "Office",      "price": 34.99,  "in_stock": True},
    ],
]

for day_offset, batch in enumerate(batches):
    file_path = f"{volume_path}/products_2026-02-{1 + day_offset:02d}.json"
    content = "\n".join(json.dumps(r) for r in batch)
    dbutils.fs.put(file_path, content, overwrite=True)

display(dbutils.fs.ls(volume_path))

The source data above has the exact schema we defined on the target table — same column names, same types. COPY INTO compares incoming data to the target table's schema, so matching them up front avoids hitting evolution-mode questions on the first run. We'll deliberately introduce a mismatch later to see what happens.

In [0]:
%sql
COPY INTO certprep.ingestion.products
FROM (
  SELECT 
    CAST(product_id AS INT) AS product_id,
    product_name,
    category,
    CAST(price AS DECIMAL(10, 2)) AS price,
    in_stock
  FROM '/Volumes/certprep/ingestion/copy_into_files'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'false');

In [0]:
%sql
SELECT * FROM certprep.ingestion.products ORDER BY product_id;

In [0]:
%sql
-- Inspect the table's transaction log to see what COPY INTO recorded
DESCRIBE HISTORY certprep.ingestion.products;

COPY INTO records each ingestion as an operation in the target Delta table's transaction log. Unlike Auto Loader, which keeps file-tracking 
state in a separate checkpoint directory, COPY INTO's state lives in the target table itself. The transaction log is what COPY INTO consults 
to determine which files have already been ingested on subsequent runs.

COPY INTO supports two forms:

1. Direct: `COPY INTO target FROM 'path' FILEFORMAT = ...`
   Used when source data exactly matches the target schema.

2. SELECT form: `COPY INTO target FROM (SELECT ... FROM 'path') FILEFORMAT = ...`
   Used when transformation is needed on the way in — casts, renames, 
   derived columns, filtering. This is the more common production form 
   because source schemas rarely match target schemas exactly.

The file-tracking behavior is identical for both forms. The transformation 
happens during the read; idempotency is unchanged.

In [0]:
%sql
-- Run again to test idempotency 
COPY INTO certprep.ingestion.products
FROM (
  SELECT 
    CAST(product_id AS INT) AS product_id,
    product_name,
    category,
    CAST(price AS DECIMAL(10, 2)) AS price,
    in_stock
  FROM '/Volumes/certprep/ingestion/copy_into_files'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'false');

In [0]:
%sql
-- Should still be 6
SELECT COUNT(*) FROM certprep.ingestion.products;

In [0]:
%sql
DESCRIBE HISTORY certprep.ingestion.products;

Running the COPY INTO again did not reinsert the file. The rerun was surprisingly not reflected in the `DESCRIBE HISTORY` results. 

COPY INTO commits a transaction only when files are actually ingested. A no-op run completes successfully without writing to the transaction log. DESCRIBE HISTORY therefore won't show idempotent re-runs — only the original ingestion and any subsequent runs that found new files.

COPY INTO listed the source directory and found three files. It then checked the transaction log of the target table and saw that all three of those files had been ingested in the previous run. So it ingested nothing.

This is the idempotency guarantee: running the same COPY INTO statement multiple times produces the same result as running it once. It's the property that lets you safely retry a failed job, or run the same command on a schedule without worrying about duplicates.

Contrast with Auto Loader: both mechanisms achieve the same outcome (exactly-once-per-file ingestion), but the state lives in different places. Auto Loader uses a separate checkpoint directory; COPY INTO uses the target table's transaction log.

In [0]:
# create a new file
new_batch = [
    {"product_id": 7, "product_name": "Yoga Mat",      "category": "Fitness", "price": 24.99, "in_stock": True},
    {"product_id": 8, "product_name": "Dumbbells Set", "category": "Fitness", "price": 79.50, "in_stock": False},
]

dbutils.fs.put(
    f"{volume_path}/products_2026-02-04.json",
    "\n".join(json.dumps(r) for r in new_batch),
    overwrite=True,
)

display(dbutils.fs.ls(volume_path))

In [0]:
%sql
COPY INTO certprep.ingestion.products
FROM (
  SELECT 
    CAST(product_id AS INT) AS product_id,
    product_name,
    category,
    CAST(price AS DECIMAL(10, 2)) AS price,
    in_stock
  FROM '/Volumes/certprep/ingestion/copy_into_files'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'false');

In [0]:
%sql
SELECT COUNT(*) FROM certprep.ingestion.products;
-- Should be 8 now

In [0]:
%sql
SELECT * FROM certprep.ingestion.products WHERE category = 'Fitness';

COPY INTO is idempotent at the file level. Running the same statement twice ingests files only once. When new files appear in the source directory, the same statement picks them up incrementally — no syntax change required.

The mechanism: on each run, COPY INTO lists source files and compares to the target table's transaction log. Only files not already recorded are processed. The transaction log is the source of truth for "what has been ingested," analogous to Auto Loader's checkpoint directory but living inside the target table itself.

This is why COPY INTO works well for scheduled batch loads: schedule the same statement to run nightly, and it handles whatever's new each run without duplicating prior data.

In [0]:
# create a new file with a new column to cause a schema mismatch then evolve the schema

evolved_batch = [
    {"product_id": 9,  "product_name": "Running Shoes", "category": "Fitness", "price": 89.99, "in_stock": True, "weight_oz": 12.5},
    {"product_id": 10, "product_name": "Hiking Boots",  "category": "Fitness", "price": 145.00,"in_stock": True, "weight_oz": 24.0},
]

dbutils.fs.put(
    f"{volume_path}/products_2026-02-05.json",
    "\n".join(json.dumps(r) for r in evolved_batch),
    overwrite=True,
)

display(dbutils.fs.ls(volume_path))

In [0]:
%sql
-- Run the copy into without mergeSchema to see the default behavior
COPY INTO certprep.ingestion.products
FROM (
  SELECT 
    CAST(product_id AS INT) AS product_id,
    product_name,
    category,
    CAST(price AS DECIMAL(10, 2)) AS price,
    in_stock
  FROM '/Volumes/certprep/ingestion/copy_into_files'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'false');

In [0]:
%sql
SELECT COUNT(*) FROM certprep.ingestion.products;
-- Should be 10

In [0]:
%sql
SELECT * FROM certprep.ingestion.products WHERE product_id IN (9, 10);

Note that while new rows were copied, the `weight_oz` column was not copied into the table. COPY INTO only copies in columns that match the predefined schema by default.

In [0]:
%sql
DROP TABLE certprep.ingestion.products;

CREATE TABLE certprep.ingestion.products (
  product_id BIGINT,
  product_name STRING,
  category STRING,
  price DOUBLE,
  in_stock BOOLEAN
);

In [0]:
%sql
COPY INTO certprep.ingestion.products
FROM '/Volumes/certprep/ingestion/copy_into_files'
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true');

JSON doesn't carry rich type information. When Spark infers types from JSON, it makes conservative choices — wider integer types, DOUBLE for decimals — that often don't match handcrafted target schemas. The direct form of COPY INTO requires the inferred source schema to be assignable to the target without coercion. When they don't match, either widen the target schema, use the SELECT form to cast explicitly, or pass schema hints to the JSON parser.

In [0]:
%sql
DESCRIBE TABLE certprep.ingestion.products;

In [0]:
%sql
SELECT * FROM certprep.ingestion.products ORDER BY product_id;

In [0]:
%sql
-- clean up and verify
DROP TABLE IF EXISTS certprep.ingestion.products;
DROP VOLUME IF EXISTS certprep.ingestion.copy_into_files;

SHOW TABLES IN certprep.ingestion;
SHOW VOLUMES IN certprep.ingestion;

1. COPY INTO and Auto Loader both provide incremental, idempotent file ingestion to Delta tables. Where does each one store the state that tracks which files have been ingested? What's the practical implication of that difference?

2. You run the same COPY INTO statement twice in a row, with no new files in the source directory between runs. What does the second run do? What appears in the transaction log? Why?

3. A team has a nightly job that ingests files from a directory into a Delta table. The job ran successfully last night but a network blip caused the orchestrator to retry it three times before the final successful run. How many copies of last night's data are in the table? Why?

4. COPY INTO supports both a direct form (`COPY INTO target FROM 'path' FILEFORMAT = ...`) and a SELECT form (`COPY INTO target FROM (SELECT ... FROM 'path') FILEFORMAT = ...`). When would you reach for each? Give a concrete scenario for each form.

5. A new column appears in source JSON files. The COPY INTO statement uses the direct form with no COPY_OPTIONS specified. What happens? How does this differ from the same scenario using the SELECT form that didn't include the new column?

6. Your team is choosing between COPY INTO and Auto Loader for a new pipeline. The source is an S3 prefix that receives roughly 200 files per day. The pipeline runs on a nightly schedule. The team prefers SQL over PySpark. Which would you recommend, and what would push you to recommend the other instead?

1. COPY INTO stores file tracking in the target table's history itself (accessible via DESCRIBE HISTORY) and Auto Loader stores file tracking in a checkpoint directory (within the source volume in this case). One practical implication in this difference is that the Auto Loader's checkpoint directory must be protected from careless deletion, as such would allow duplicate data to be ingested should old files still rest in the source location. Auto Loader's history can be lost without losing the table, while COPY INTO's history can never be lost without losing the table.
2. With no file additions to the source location, running COPY INTO again simply does not ingest the data again. The table history tracks that these files have already been ingested. The transaction log is not updated to reflect the second run as no transactions have actually modified the table.
3. There should be only 1 copy of the data. 3 failures followed by retries would not commit any data until a complete successful run can ingest the data in full. They are atomic transactions (ACID). Once a successful run has occurred, any subsequent reruns will not reupload the data, as the ingestion history is tracked within the table's history.
4. Direct form COPY INTO is appropriate when the source schema matches the target table's scehma. Direct form works when the inferred source schema is type-compatible with the target schema. SELECT form is approriate when transformations are necessary before loading. Transformations can include data type castings, renaming columns with an alias, derived columns, etc. Direct form when the source's natural inference produces types your target accepts; SELECT form when it doesn't, or when you want to transform.
5. The direct form will ingest row values for all columns except the new columns. The same goes for SELECT form, but because the subquery does not actually select the new column, thus it is not ingested. 
6. If the team prefers SQL-first, then they should choose COPY INTO as that is what it is built for and 200 files/day is no issue. Auto Loader a better option if the volume is likely to grow into the millions, as the resource cost of listing that many files increases greatly. Also, if the team needs to transition to lower-latency ingestion, such as hourly, Auto Loader supports streaming. More complex transformations are also expressed better in the dataframe API with Auto Loader than in COPY INTO's SELECT form.